# Experiment: Task 2 Colab Runner (Scratch vs Fine-tune on Oxford-IIIT Pet)

这个 notebook 的目标不是展示，而是**稳定把结果跑出来并落到 Google Drive**。

固定实验：
- `DenseNet121 scratch`
- `DenseNet121 finetune`
- `ResNeXt50_32x4d scratch`
- `ResNeXt50_32x4d finetune`

使用方式：
1. 先运行 setup cell
2. 再运行 GPU 检查 cell
3. 先跑 smoke test
4. 确认 smoke 结果落到 Drive
5. 再跑 full run
6. 最后汇总并生成 `main.tex`


In [ ]:
from __future__ import annotations

import os
import shlex
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('This notebook is intended for Google Colab.')

REPO_URL = 'https://github.com/sidchy/math3705m-neural-networks-assignments.git'
REPO_WEB_URL = REPO_URL.removesuffix('.git')

WORKSPACE = Path('/content/drive/MyDrive/task2_pet_workspace')
PROJECT_DIR = WORKSPACE / 'project'
DATA_DIR = WORKSPACE / 'data'
RUNS_DIR = WORKSPACE / 'runs'
SMOKE_DIR = WORKSPACE / 'smoke_runs'
REPORT_DIR = PROJECT_DIR / 'task2' / 'report'

DEVICE = 'cuda'
NUM_WORKERS = 2
PRESETS = [
    'densenet121_scratch',
    'densenet121_finetune',
    'resnext50_32x4d_scratch',
    'resnext50_32x4d_finetune',
]

def sh(cmd: str, cwd: Path | None = None) -> None:
    print(f'+ {cmd}')
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=True,
        text=True,
        bufsize=1,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}')

def ls_tree(path: Path, max_depth: int = 2) -> None:
    print(f'Listing: {path}')
    if not path.exists():
        print('  <missing>')
        return
    for item in sorted(path.rglob('*')):
        depth = len(item.relative_to(path).parts)
        if depth <= max_depth:
            print(' ', item)

WORKSPACE.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

if not PROJECT_DIR.exists():
    sh(f'git clone {shlex.quote(REPO_URL)} {shlex.quote(str(PROJECT_DIR))}')
else:
    sh('git pull --ff-only', cwd=PROJECT_DIR)

sh(f'{shlex.quote(sys.executable)} -m pip install -q -r task2/requirements-colab.txt', cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)

print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR =', DATA_DIR)
print('SMOKE_DIR =', SMOKE_DIR)
print('RUNS_DIR =', RUNS_DIR)
print('REPORT_DIR =', REPORT_DIR)


In [ ]:
import torch
import torchvision

print('cwd =', Path.cwd())
print('torch =', torch.__version__)
print('torchvision =', torchvision.__version__)
print('cuda available =', torch.cuda.is_available())
print('run_all exists =', (PROJECT_DIR / 'task2' / 'run_all.py').exists())
print('train exists =', (PROJECT_DIR / 'task2' / 'train.py').exists())

if torch.cuda.is_available():
    sh('nvidia-smi')
else:
    raise RuntimeError('GPU is not enabled. Please switch Runtime -> Change runtime type -> GPU.')


## Step 0: 先看 Drive 里当前已经有什么

如果昨晚跑过，这一步能告诉你哪些结果已经落盘，避免重复跑。


In [ ]:
print('SMOKE_DIR exists =', SMOKE_DIR.exists())
print('RUNS_DIR exists =', RUNS_DIR.exists())
print('REPORT_DIR exists =', REPORT_DIR.exists())

print('\nSmoke runs:')
for p in sorted(SMOKE_DIR.glob('*')):
    print(' ', p)

print('\nFormal runs:')
for p in sorted(RUNS_DIR.glob('*')):
    print(' ', p)

print('\nReport files:')
for p in sorted(REPORT_DIR.glob('*')):
    print(' ', p)


## Step 1: Smoke Test

这一步只跑 1 epoch，用来确认：数据下载、预训练权重、checkpoint、图片导出、Google Drive 写盘都正常。

只在 smoke 不完整时重新执行。已经有 4 组 smoke 目录的话，可以跳过到 Step 2。


In [ ]:
sh(
    f'{shlex.quote(sys.executable)} -u task2/run_all.py '
    f'--data-root {shlex.quote(str(DATA_DIR))} '
    f'--output-root {shlex.quote(str(SMOKE_DIR))} '
    f'--device {shlex.quote(DEVICE)} '
    f'--num-workers {NUM_WORKERS} '
    '--smoke-test',
    cwd=PROJECT_DIR,
)


In [ ]:
print('Smoke directory contents:')
for run_dir in sorted(SMOKE_DIR.glob('*')):
    print('\n==', run_dir.name)
    if run_dir.is_dir():
        for f in sorted(run_dir.glob('*')):
            print(' ', f.name)


## Step 2: 单组正式训练试跑（推荐）

这一步只做一件事：确认 **正式输出会写进 `RUNS_DIR`**。如果昨晚 full run 没留下任何目录，先跑这一组最稳。

如果你已经在 `RUNS_DIR` 里看到了完整正式目录，也可以跳过到 Step 3。


In [ ]:
sh(
    f'{shlex.quote(sys.executable)} -u task2/train.py '
    '--preset densenet121_scratch '
    f'--data-root {shlex.quote(str(DATA_DIR))} '
    f'--output-root {shlex.quote(str(RUNS_DIR))} '
    f'--device {shlex.quote(DEVICE)} '
    f'--num-workers {NUM_WORKERS}',
    cwd=PROJECT_DIR,
)


In [ ]:
print('Current RUNS_DIR contents:')
for p in sorted(RUNS_DIR.glob('*')):
    print(' ', p)


## Step 3: Full Run

这一步顺序跑 4 组正式实验，并在总训练时间不足 7200 秒时自动给 scratch 组补训练。

运行后不要立即关页面，先确认 `RUNS_DIR` 已经出现正式目录。


In [ ]:
sh(
    f'{shlex.quote(sys.executable)} -u task2/run_all.py '
    f'--data-root {shlex.quote(str(DATA_DIR))} '
    f'--output-root {shlex.quote(str(RUNS_DIR))} '
    f'--device {shlex.quote(DEVICE)} '
    f'--num-workers {NUM_WORKERS}',
    cwd=PROJECT_DIR,
)


In [ ]:
print('Formal runs after full run:')
for run_dir in sorted(RUNS_DIR.glob('*')):
    if run_dir.is_dir():
        print('\n==', run_dir.name)
        for f in sorted(run_dir.glob('*')):
            print(' ', f.name)


## Step 4: 汇总并生成报告源码


In [ ]:
sh(
    f'{shlex.quote(sys.executable)} -u task2/summarize_runs.py '
    f'--runs-root {shlex.quote(str(RUNS_DIR))} '
    f'--report-dir {shlex.quote(str(REPORT_DIR))}',
    cwd=PROJECT_DIR,
)

sh(
    f'{shlex.quote(sys.executable)} -u task2/make_report.py '
    f'--summary-json {shlex.quote(str(REPORT_DIR / "summary.json"))} '
    f'--outdir {shlex.quote(str(REPORT_DIR))} '
    f'--repo-url {shlex.quote(REPO_WEB_URL)}',
    cwd=PROJECT_DIR,
)


## Step 5: 验收检查

这一步必须确认：
- `rows == 4`
- `total_train_seconds >= 7200`


In [ ]:
import json

summary = json.loads((REPORT_DIR / 'summary.json').read_text(encoding='utf-8'))
print('rows =', len(summary['rows']))
print('total_train_seconds =', summary['total_train_seconds'])
print()

for row in summary['rows']:
    print(
        row['display_name'],
        'test_acc =', row['test_acc'],
        'macro_f1 =', row['macro_f1'],
        'train_seconds =', row['train_seconds'],
        'best_epoch =', row['best_epoch'],
    )


In [ ]:
from IPython.display import display
from PIL import Image

for name in [
    'accuracy_comparison.png',
    'efficiency_tradeoff.png',
    'densenet_curves.png',
    'resnext_curves.png',
    'best_predictions_grid.png',
    'best_top_confusions.png',
]:
    image_path = REPORT_DIR / 'figs' / name
    print(name, image_path.exists())
    if image_path.exists():
        display(Image.open(image_path))


## Step 6: 打包下载

只把报告目录和主要结果文件打包回本地，避免 checkpoint 太大。


In [ ]:
import zipfile

bundle_path = WORKSPACE / 'task2_report_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in REPORT_DIR.rglob('*'):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(PROJECT_DIR))
    for path in RUNS_DIR.rglob('*'):
        if path.is_file() and path.suffix in {'.json', '.csv', '.png'}:
            archive.write(path, arcname=path.relative_to(PROJECT_DIR))

print('Created:', bundle_path)
